# Exploration du jeu de données sur les mesures de radiations prélevées

Ce notebook sert d'exploration au jeu de données d'openradiation pour pouvoir comprendre la structure des trois fichiers du dossier tar mise à jour une fois par jour.

On commencera dans un premier temps par une ingestion de ce fichier et ensuite extraire les trois fichiers CSV conservées et les explorer.

Cela nous permettra de déterminer le chemin à suivre pour l'ingestion, le traitement et l'analyse de ces données.

In [1]:
# Chargement des librairies
import requests
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import tarfile
from notebookutils import mssparkutils # La gestion du lakehouse
import os
from pathlib import Path

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 5, Finished, Available, Finished, False)

# Capturer le jeu de données via l'url suivante

https://request.openradiation.net/openradiation_dataset.tar.gz

In [2]:
# Définition de l'url de requête à appliquer
URL = (
    "https://request.openradiation.net/"
    "openradiation_dataset.tar.gz"
)


StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 6, Finished, Available, Finished, False)

In [3]:
# Requête Get et récupération de la réponse
response = requests.get(
        URL,
        headers={"User-Agent": "Python OpenRadiation downloader"},
    )

print(f"Requête GET URL fichier tar à récupérer : {URL}")
status = response.status_code

if status != 200:
            raise RuntimeError(f"HTTP inattendu : {status}")

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 7, Finished, Available, Finished, False)

Requête GET URL fichier tar à récupérer : https://request.openradiation.net/openradiation_dataset.tar.gz


Enregistrons le fichier résultant dans notre lakehouse (openradiation_lakehouse)

In [4]:
LAKEHOUSE_PATH = "Files/OpenRadiation" # Destination de mon fichier

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 8, Finished, Available, Finished, False)

In [5]:
# Créer le dossier de destination
mssparkutils.fs.mkdirs(LAKEHOUSE_PATH)

# Écrire le contenu binaire dans un fichier temporaire local
local_path = "/tmp/openradiation_dataset.tar.gz"

with open(local_path, "wb") as f:
    f.write(response.content)

# Copier le fichier dans le Lakehouse
mssparkutils.fs.cp(
    f"file:{local_path}",
    LAKEHOUSE_PATH+"/openradiation_dataset.tar.gz"
)

print(f"Archive enregistrée dans le Lakehouse : {LAKEHOUSE_PATH}")
print(f"Taille : {len(response.content):,} octets")

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 9, Finished, Available, Finished, False)

Archive enregistrée dans le Lakehouse : Files/OpenRadiation
Taille : 449,909,469 octets


Le fichier a bien été enregistré.

# Extraction des fichiers dans le fichier compressé 

Maintenant que nous avons collecté le fichier compressé, extrayons ces différents fichiers.

Définissons d'abord les chemins temporaires et de destinations.

In [6]:
# Répertoire temporaire local pour les fichiers CSV
TEMP_DIR = "/tmp/openradiation_extract"

# Destination finale dans le Lakehouse
LAKEHOUSE_DIR = "Files/OpenRadiation/extracted"

os.makedirs(TEMP_DIR, exist_ok=True)
mssparkutils.fs.mkdirs(LAKEHOUSE_DIR)

# Extraction des fichiers CSV uniquement
csv_files = []

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 10, Finished, Available, Finished, False)

Lisons le fichier compressé et extraire les fichiers CSV qui s'y trouve.

In [7]:
# Chemin pour lire sera celui-ci
print(os.listdir("/lakehouse/default/Files/OpenRadiation/"))
ARCHIVE_PATH="/lakehouse/default/Files/OpenRadiation/openradiation_dataset.tar.gz"

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 11, Finished, Available, Finished, False)

['extracted', 'openradiation_dataset.tar.gz']


In [8]:
# Ouverture du fichier compressé et récupération des fichiers CSV
with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
    for member in tar.getmembers():
        if not member.isfile():
            continue

        # Ne retenir que les CSV
        if not member.name.lower().endswith(".csv"):
            continue

        # Éviter les chemins dangereux dans l'archive
        filename = Path(member.name).name
        if not filename:
            continue

        source = tar.extractfile(member)
        if source is None:
            continue

        local_file = Path(TEMP_DIR) / filename

        with source, local_file.open("wb") as output:
            output.write(source.read())

        csv_files.append(local_file)

# Copier les CSV vers le Lakehouse
for csv_file in csv_files:
    destination = f"{LAKEHOUSE_DIR}/{csv_file.name}"

    mssparkutils.fs.cp(
        f"file:{csv_file}",
        destination
    )

    print(f"Fichier extrait : {destination}")

print(f"\nNombre de fichiers CSV extraits : {len(csv_files)}")

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 12, Finished, Available, Finished, False)

Fichier extrait : Files/OpenRadiation/extracted/measurements.csv
Fichier extrait : Files/OpenRadiation/extracted/tags.csv
Fichier extrait : Files/OpenRadiation/extracted/measurements_withoutEnclosedObject.csv

Nombre de fichiers CSV extraits : 3


Maintenant que nous avons extraits les fichiers CSV, lisons les avec Spark et observons leur stucture.

# Observation des différents jeu de données

Les fichiers CSV sont maintenant extraites, il ne reste plus qu'à les lire et à comprendre leur stucture.

In [9]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

def analyser_dataframe(df: DataFrame) -> None:
    """
    Analyse un DataFrame Spark :
    - nombre de lignes et de colonnes
    - proportion de données manquantes par colonne
    - structure et types des colonnes
    """

    # 1. Nombre de lignes et de colonnes
    nb_lignes = df.count()
    nb_colonnes = len(df.columns)

    print("Nombre de lignes / colonnes", nb_lignes, "/", nb_colonnes)

    # 2. Structure des colonnes
    print("\n--- Structure des colonnes ---")
    df.printSchema()

    # 3. Données manquantes par colonne
    print("\n--- Données manquantes par colonne ---")

    if nb_lignes == 0:
        print("Le DataFrame est vide : proportions non calculables.")
        return

    expressions = []

    for colonne, type_colonne in df.dtypes:
        # NULL pour tous les types ; chaîne vide ou espaces pour les chaînes
        condition_manquante = F.col(colonne).isNull()

        if type_colonne == "string":
            condition_manquante = (
                condition_manquante
                | (F.trim(F.col(colonne)) == "")
            )

        expressions.extend([
            F.sum(
                F.when(condition_manquante, 1).otherwise(0)
            ).alias(f"{colonne}__manquants")
        ])

    resultats = df.agg(*expressions).first()

    for colonne, type_colonne in df.dtypes:
        nb_manquants = resultats[f"{colonne}__manquants"]
        proportion = nb_manquants / nb_lignes * 100

        print(
            f"Colonne : {colonne} | "
            f"Type : {type_colonne} | "
            f"Manquants : {nb_manquants} | "
            f"Proportion : {proportion:.2f} %"
        )



StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 13, Finished, Available, Finished, False)

In [10]:
def read_csv_file(spark, path):
    """
    Permets de lire un fichier CSV avec ";" comme séparation 
    et de le renvoyer en Spark DataFrame
    """
    df = (
            spark.read
            .option("header", True)
            .option("sep", ";")
            .option("inferSchema", True)
            .csv(path)
        )
    return df


StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 14, Finished, Available, Finished, False)

In [11]:
# Lecture des différents fichiers CSV
files = mssparkutils.fs.ls(LAKEHOUSE_DIR)

for file in files:
    if file.name.lower().endswith(".csv"):
        print(f"\nLecture de {file.name}")

        df = read_csv_file(spark, file.path)

        df.printSchema()
        df.show(5, truncate=False)

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 15, Finished, Available, Finished, False)


Lecture de measurements.csv
root
 |-- apparatusId: string (nullable = true)
 |-- apparatusVersion: string (nullable = true)
 |-- apparatusSensorType: string (nullable = true)
 |-- apparatusTubeType: string (nullable = true)
 |-- temperature: string (nullable = true)
 |-- value: double (nullable = true)
 |-- hitsNumber: string (nullable = true)
 |-- calibrationFunction: string (nullable = true)
 |-- startTime: string (nullable = true)
 |-- endTime: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- accuracy: double (nullable = true)
 |-- altitude: integer (nullable = true)
 |-- altitudeAccuracy: double (nullable = true)
 |-- endLatitude: double (nullable = true)
 |-- endLongitude: double (nullable = true)
 |-- endAccuracy: string (nullable = true)
 |-- endAltitude: string (nullable = true)
 |-- endAltitudeAccuracy: string (nullable = true)
 |-- deviceUuid: string (nullable = true)
 |-- devicePlatform: string (nullable = true)


Associons une variable chacune pour les observer séparément

In [12]:
measurements = read_csv_file(spark, LAKEHOUSE_DIR+"/measurements.csv")
measurements_without_enclobject = read_csv_file(spark, LAKEHOUSE_DIR+"/measurements_withoutEnclosedObject.csv")
tags = read_csv_file(spark, LAKEHOUSE_DIR+"/tags.csv")

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 16, Finished, Available, Finished, False)

## Observation de la structure pour measurements.csv

In [13]:
print("Nombre de ligne et de colonne : ", measurements.count(), " / ", len(measurements.columns))
measurements.printSchema()

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 17, Finished, Available, Finished, False)

Nombre de ligne et de colonne :  2031544  /  60
root
 |-- apparatusId: string (nullable = true)
 |-- apparatusVersion: string (nullable = true)
 |-- apparatusSensorType: string (nullable = true)
 |-- apparatusTubeType: string (nullable = true)
 |-- temperature: string (nullable = true)
 |-- value: double (nullable = true)
 |-- hitsNumber: string (nullable = true)
 |-- calibrationFunction: string (nullable = true)
 |-- startTime: string (nullable = true)
 |-- endTime: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- accuracy: double (nullable = true)
 |-- altitude: integer (nullable = true)
 |-- altitudeAccuracy: double (nullable = true)
 |-- endLatitude: double (nullable = true)
 |-- endLongitude: double (nullable = true)
 |-- endAccuracy: string (nullable = true)
 |-- endAltitude: string (nullable = true)
 |-- endAltitudeAccuracy: string (nullable = true)
 |-- deviceUuid: string (nullable = true)
 |-- devicePlatform: string

### Description du jeu de données 


<table>
<tr><th>API Name</th><th>Type</th><th>Available for request API</th><th>Available for submit API</th><th>Description</th></tr>
<tr><td>apparatusId</td><td>String</td><td></td><td></td><td>Unique sensor identifier</td></tr>
<tr><td>apparatusVersion</td><td>String</td><td> </td><td> </td><td>Sensor firmware version</td></tr>
<tr><td>apparatusSensorType</td><td>String</td><td> </td><td> </td><td>Sensor type : geiger, photodiode</td></tr>
<tr><td>apparatusTubeType</td><td>String</td><td> </td><td> </td><td>Tube identification (only if apparatusSensorType = geiger)</td></tr>
<tr><td>temperature</td><td>Integer</td><td></td><td></td><td>Temperature (°C) </td></tr>
<tr><td>value</td><td>Real</td><td>*</td><td>Mandatory</td><td>Dose rate (µSv/h)</td></tr>
<tr><td>hitsNumber</td><td>Integer</td><td></td><td></td><td>Counts measured during the whole measurement period. cps (counts per second) used to evaluate dose rate with the calibration function will be : cps = hitsNumber / (endTime - startTime)</td></tr>
<tr><td>calibrationFunction</td><td>String</td><td></td><td></td><td>Calibration function used to calculate µSv/h from cps (counts per second). Format should be inline with these symbols : cps 0-9. -+/*^() max,. Example : 7.543*(cps-0.02)^2+0.001*(cps-0.02)</td></tr>
<tr><td>startTime</td><td>Timestamp</td><td>*</td><td>Mandatory</td><td>Date of the beginning of the measurement (ISO GMT)</td></tr>
<tr><td>endTime</td><td>Timestamp</td><td></td><td> </td><td>Date of the end of the measurement (ISO GMT)</td></tr>
<tr><td>latitude</td><td>Real</td><td>*</td><td>Mandatory</td><td>latitude</td></tr>
<tr><td>longitude</td><td>Real</td><td>*</td><td>Mandatory</td><td>longitude</td></tr>
<tr><td>accuracy</td><td>Real</td><td></td><td> </td><td>Position accuracy in meters</td></tr>
<tr><td>altitude</td><td>Integer</td><td></td><td> </td><td>Altitude above sea in meters</td></tr>
<tr><td>altitudeAccuracy</td><td>Real</td><td></td><td> </td><td>Altitude accuracy in meters</td></tr>
<tr><td>endLatitude</td><td>Real</td><td></td><td></td><td>latitude at the end of the measurement</td></tr>
<tr><td>endLongitude</td><td>Real</td><td></td><td></td><td>longitude at the end of the measurement</td></tr>
<tr><td>endAccuracy</td><td>Real</td><td></td><td></td><td>Position accuracy in meters at the end of the measurement</td></tr>
<tr><td>endAltitude</td><td>Integer</td><td></td><td></td><td>Altitude above sea in meters at the end of the measurement</td></tr>
<tr><td>endAltitudeAccuracy</td><td>Real</td><td></td><td></td><td>Altitude accuracy in meters at the end of the measurement</td></tr>
<tr><td>deviceUuid</td><td>String</td><td></td><td> </td><td>Smartphone device UUID  (see http://plugins.cordova.io/#/package/org.apache.cordova.device)</td></tr>
<tr><td>devicePlatform</td><td>String</td><td></td><td> </td><td>Smartphone device platform</td></tr>
<tr><td>deviceVersion</td><td>String</td><td></td><td> </td><td>Smartphone device OS version</td></tr>
<tr><td>deviceModel</td><td>String</td><td></td><td> </td><td>Smartphone device model</td></tr>
<tr><td>reportUuid</td><td>String</td><td>*</td><td>Mandatory</td><td>Unique measurement UUID (UUIDv4 format) client-side generated</td></tr>
<tr><td>manualReporting</td><td>Boolean</td><td></td><td></td><td>Manual Reporting : true, false (default:true). False if the data is not entered by a human being</td></tr>
<tr><td>organisationReporting</td><td>String</td><td></td><td></td><td>Software version (sample:openradiation_v1)</td></tr>
<tr><td>reportContext</td><td>String</td><td>Never</td><td></td><td>Report context : emergency, routine, exercise, test (default:test). test:data are not registrated but you can test api use, emergency and exercise:not used,routine:you should use this one !</td></tr>
<tr><td>description</td><td>String</td><td></td><td></td><td>Free description (only if userId is specified)</td></tr>
<tr><td>measurementHeight</td><td>Integer</td><td></td><td></td><td>Measurement height above the ground (in meters). It is measurement height from the ground (and not altitude above sea level).</td></tr>
<tr><td>tags</td><td>Json array of string</td><td></td><td></td><td>Free tags list [tag1 ; tag2] (only if userId is specified)</td></tr>
<tr><td>enclosedObject</td><td>String</td><td></td><td></td><td>Base64 encoded Image. The size shoudn't exceeded 1mb and format should be closed from 600*800 pixels (width * height). The value should be a data URI scheme 'data:image/<subtype>;base64,<data>'. (only if userId is specified)</td></tr>
<tr><td>userId</td><td>String</td><td></td><td></td><td>Openradiation.org user id</td></tr>
<tr><td>userPwd</td><td>String</td><td>Never</td><td></td><td>Openradiation.org plain text password (mandatory if userId is specified)</td></tr>
<tr><td>measurementEnvironment</td><td>String</td><td></td><td></td><td>Measurement environment : countryside, city, ontheroad, inside, plane (if plane, qualification is set to plane)</td></tr>
<tr><td>rain</td><td>Boolean</td><td></td><td></td><td>Rain : true if it rains during the measurement</td></tr>
<tr><td>flightNumber</td><td>String</td><td></td><td></td><td>if measurementEnvironment is plane, flightNumber of the commercial flight in capital letters (AITA code followed by number, example: AF179)</td></tr>
<tr><td>seatNumber</td><td>String</td><td></td><td></td><td>if measurementEnvironment is plane, seatNumber in capital letters with row number first (example: 14C)</td></tr>
<tr><td>windowSeat</td><td>Boolean</td><td></td><td></td><td>if measurementEnvironment is plane, windowSeat : true if the seat where is the sensor is next to the window</td></tr>
<tr><td>storm</td><td>Boolean</td><td></td><td></td><td>Storm : true if storm crossing during the measurement</td></tr>
<tr><td>flightId</td><td>Integer</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber setted, measurement is associated to a flightId (flightId permits to retrieve all measurements in a same flight)</td></tr>
<tr><td>refinedLatitude</td><td>Real</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber recognize, latitude is refined with the track of the plane</td></tr>
<tr><td>refinedLongitude</td><td>Real</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber recognize, longitude is refined with the track of the plane</td></tr>
<tr><td>refinedAltitude</td><td>Integer</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber recognize, altitude is refined with the track of the plane</td></tr>
<tr><td>refinedEndLatitude</td><td>Real</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber recognize, endLatitude is refined with the track of the plane</td></tr>
<tr><td>refinedEndLongitude</td><td>Real</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber recognize, endLongitude is refined with the track of the plane</td></tr>	
<tr><td>refinedEndAltitude</td><td>Integer</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber recognize, endAltitude is refined with the track of the plane</td></tr>
<tr><td>departureTime</td><td>Timestamp</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber recognize, actual departure time of the plane</td></tr>
<tr><td>arrivalTime</td><td>Timestamp</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber recognize, actual arrival time of the plane</td></tr>
<tr><td>airportOrigin</td><td>String</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber recognize, airport origin (AITA) of the plane</td></tr>
<tr><td>airportDestination</td><td>String</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber recognize, airport destination (AITA) of the plane</td></tr>	
<tr><td>aircraftType</td><td>String</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber recognize, aircraft type of the plane</td></tr>
<tr><td>firstLatitude</td><td>Real</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber setted, latitude of the first measurement or latitude of the origin airport</td></tr>
<tr><td>firstLongitude</td><td>Real</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber setted, longitude of the first measurement or longitude of the origin airport</td></tr>
<tr><td>midLatitude</td><td>Real</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber setted, latitude picked up from a measurement or in the middle of the plane track</td></tr>
<tr><td>midLongitude</td><td>Real</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber setted, longitude picked up from a measurement or in the middle of the plane track</td></tr>
<tr><td>lastLatitude</td><td>Real</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber setted, latitude of the last measurement or latitude of the destination airport</td></tr>
<tr><td>lastLongitude</td><td>Real</td><td></td><td>No, determinated by the API</td><td>if measurementEnvironment is plane and flightNumber setted, longitude of the last measurement or longitude of the destination airport</td></tr>
<tr><td>dateAndTimeOfCreation</td><td>Timestamp</td><td></td><td>No, but always determinated by the API</td><td>Date of registration in the database</td></tr>
<tr><td>qualification</td><td>String</td><td>*</td><td>No, determinated by the API or the website</td><td>plane, wrongmeasurement, temporarysource, groundlevel</td></tr>
<tr><td>qualificationVotesNumber</td><td>Integer</td><td></td><td>No, determinated by the API or the website</td><td>qualification Votes Number</td></tr>
<tr><td>reliability</td><td>Integer</td><td></td><td>No, but always determinated by the API and never modified</td><td>Estimated reliability that the measurement is correct. Calculated when submitted to the API as following : +1 for each filled field, + min(30,HitsNumber) if HitsNumber not null, +10 if userId not null, +20 if ManualReporting=false, +20 if MeasurementEnvironment=countryside / +10 if MeasurementEnvironment=city or ontheroad, +10 if MeasurementHeight=1. Expecting > 78 (if not qualification is set to mustbeverified and qualificationVotesNumber is set to 0)</td></tr>
<tr><td>atypical</td><td>Boolean</td><td>*</td><td>No, but always determinated by the API and never modified</td><td>atypical if value is not representative of an environnemental measure. No if value < 0.2 (but we should compare value to an estimated local reference ...), yes otherwise</td></tr>
</table>


Observons la proportion des données manquantes sur chaque colonne.

In [14]:
analyser_dataframe(measurements)

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 18, Finished, Available, Finished, False)

Nombre de lignes / colonnes 2031544 / 60

--- Structure des colonnes ---
root
 |-- apparatusId: string (nullable = true)
 |-- apparatusVersion: string (nullable = true)
 |-- apparatusSensorType: string (nullable = true)
 |-- apparatusTubeType: string (nullable = true)
 |-- temperature: string (nullable = true)
 |-- value: double (nullable = true)
 |-- hitsNumber: string (nullable = true)
 |-- calibrationFunction: string (nullable = true)
 |-- startTime: string (nullable = true)
 |-- endTime: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- accuracy: double (nullable = true)
 |-- altitude: integer (nullable = true)
 |-- altitudeAccuracy: double (nullable = true)
 |-- endLatitude: double (nullable = true)
 |-- endLongitude: double (nullable = true)
 |-- endAccuracy: string (nullable = true)
 |-- endAltitude: string (nullable = true)
 |-- endAltitudeAccuracy: string (nullable = true)
 |-- deviceUuid: string (nullable = true)
 |

Nous observons dans nos colonnes les plus importantes une présence de données manquantes :
- reportUuid : identifiant de la mesure (0.30%)
- startTime : date et temps du début de la mesure (0.30%)
- endTime : date et temps de la fin de la mesure (8.42%)
- dataAndTimeOfCreation : date et temps de la création de la mesure (0.74%)
- atypical : si la mesure est atypique (0.74%)

Le reste de ces colonnes concernent surtout des informations enregistrées en plus.

On y retrouve notamment les informations sur l'appareil, quelques informations météorologiques si renseignées, les vols concernées pour les mesures aux vols seulement et les coordonnées de la mesure du début à la fin ave ceux réajustées.

On observe donc beaucoup de colonnes dont la plupart ont une proportion de données manquantes élevées dû qu'il existe deux types de mesures (au sol et aérien).

## Observation de la structure pour measurements_withoutEnclosedObject.csv

In [15]:
analyser_dataframe(measurements_without_enclobject)

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 19, Finished, Available, Finished, False)

Nombre de lignes / colonnes 2031545 / 59

--- Structure des colonnes ---
root
 |-- apparatusId: string (nullable = true)
 |-- apparatusVersion: string (nullable = true)
 |-- apparatusSensorType: string (nullable = true)
 |-- apparatusTubeType: string (nullable = true)
 |-- temperature: integer (nullable = true)
 |-- value: string (nullable = true)
 |-- hitsNumber: string (nullable = true)
 |-- calibrationFunction: string (nullable = true)
 |-- startTime: string (nullable = true)
 |-- endTime: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- accuracy: double (nullable = true)
 |-- altitude: double (nullable = true)
 |-- altitudeAccuracy: double (nullable = true)
 |-- endLatitude: double (nullable = true)
 |-- endLongitude: string (nullable = true)
 |-- endAccuracy: string (nullable = true)
 |-- endAltitude: string (nullable = true)
 |-- endAltitudeAccuracy: string (nullable = true)
 |-- deviceUuid: string (nullable = true)
 |

Le même jeu de données que `measurements.csv` mais avec une seule colonne en moins et quelques lignes de plus.

## Observation de la structure pour le fichier tags.csv

In [16]:
analyser_dataframe(tags)

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 20, Finished, Available, Finished, False)

Nombre de lignes / colonnes 2375028 / 2

--- Structure des colonnes ---
root
 |-- reportUuid: string (nullable = true)
 |-- tag: string (nullable = true)


--- Données manquantes par colonne ---
Colonne : reportUuid | Type : string | Manquants : 0 | Proportion : 0.00 %
Colonne : tag | Type : string | Manquants : 0 | Proportion : 0.00 %


In [17]:
tags.show()

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 21, Finished, Available, Finished, False)

+--------------------+--------------------+
|          reportUuid|                 tag|
+--------------------+--------------------+
|00000000-0000-000...|              icohup|
|00000000-0000-000...|serie_46be4e4ce77...|
|0000046a-3759-4da...|series_0c81ce7984...|
|00000b30-1748-4ab...|series_e3ae9959d8...|
|00003f37-7db6-453...|series_28a15ac1f5...|
|00004030-ddc2-488...|series_f7a95040a6...|
|000056cd-6f32-420...|series_bd86100b02...|
|00007ff4-1331-495...|series_99bae0ec4b...|
|0000e6ad-af08-4d9...| arrivée: barcelone |
|0000e6ad-af08-4d9...|       départ: paris|
|0000e6ad-af08-4d9...|serie_35b93e05f3f...|
|00010b6a-c695-474...|        arrivée: cdg|
|00010b6a-c695-474...|         départ: yvr|
|00010b6a-c695-474...|serie_6580d7eccf9...|
|00010f85-b977-4e0...|series_e390d3e484...|
|00011286-55ea-4a6...|series_65ac709369...|
|000113b6-2cc3-4ad...|     challenge sirse|
|000113b6-2cc3-4ad...|series_9ebcb4fa63...|
|00011e66-3b7b-4ec...|series_41bee2dd6d...|
|000147e2-66df-4e3...|series_109

In [18]:
tags.select("reportUuid").head(10)

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 22, Finished, Available, Finished, False)

[Row(reportUuid='00000000-0000-0000-0000-000000000000'),
 Row(reportUuid='00000000-0000-0000-0000-000000000000'),
 Row(reportUuid='0000046a-3759-4dae-9ff8-4d138c5bc858'),
 Row(reportUuid='00000b30-1748-4ab7-8212-6b77ad67dabe'),
 Row(reportUuid='00003f37-7db6-453d-9fca-3cbafe417978'),
 Row(reportUuid='00004030-ddc2-4880-b7f6-2fa04c5678d0'),
 Row(reportUuid='000056cd-6f32-4202-83ad-f03f997ee7d7'),
 Row(reportUuid='00007ff4-1331-495b-9fc0-8f1c04e38332'),
 Row(reportUuid='0000e6ad-af08-4d98-9799-85910bb0fab1'),
 Row(reportUuid='0000e6ad-af08-4d98-9799-85910bb0fab1')]

Ici, on voit que les tags sont associées à une mesure via l'identifiant reportUuid.

In [19]:
tags.filter("reportUuid == '0000e6ad-af08-4d98-9799-85910bb0fab1'").show()

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 23, Finished, Available, Finished, False)

+--------------------+--------------------+
|          reportUuid|                 tag|
+--------------------+--------------------+
|0000e6ad-af08-4d9...| arrivée: barcelone |
|0000e6ad-af08-4d9...|       départ: paris|
|0000e6ad-af08-4d9...|serie_35b93e05f3f...|
+--------------------+--------------------+



La relation entre tags et mesures est selon le type de mesure. 

Pour une mesure au sol, il n'y aura qu'un seul tag (le nom de la mesure)
tandis que pour une mesure au vol, il faudra compter trois tags (tag de départ, d'arrivé et de la série). 

Il y faudra faire attention à la jointure entre les deux jeux de données.

# Préparation des mesures de radiations

Comme on a vu il existe beaucoup de colonnes qui ne seront pas utiles pour la suite de notre flux.

De plus, il faudra ajouter certaines colonnes comme le département, la région en France et seulement garder les mesures en France notamment. 

Pour la suite du notebook, nous allons utiliser principalement le DataFrame `measurements_without_enclobject`.

## Nettoyage des données

In [20]:
measurements_without_enclobject.printSchema()
print(len(measurements_without_enclobject.columns))

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 24, Finished, Available, Finished, False)

root
 |-- apparatusId: string (nullable = true)
 |-- apparatusVersion: string (nullable = true)
 |-- apparatusSensorType: string (nullable = true)
 |-- apparatusTubeType: string (nullable = true)
 |-- temperature: integer (nullable = true)
 |-- value: string (nullable = true)
 |-- hitsNumber: string (nullable = true)
 |-- calibrationFunction: string (nullable = true)
 |-- startTime: string (nullable = true)
 |-- endTime: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- accuracy: double (nullable = true)
 |-- altitude: double (nullable = true)
 |-- altitudeAccuracy: double (nullable = true)
 |-- endLatitude: double (nullable = true)
 |-- endLongitude: string (nullable = true)
 |-- endAccuracy: string (nullable = true)
 |-- endAltitude: string (nullable = true)
 |-- endAltitudeAccuracy: string (nullable = true)
 |-- deviceUuid: string (nullable = true)
 |-- devicePlatform: string (nullable = true)
 |-- deviceVersion: string (n

In [21]:
(measurements_without_enclobject.withColumns(
                                {
                                    "value" : measurements_without_enclobject.value.try_cast("float"),
                                    "startTime" : F.to_timestamp("startTime", "yyyy-MM-dd'T'HH:mm:ssX"),
                                    "endTime" : F.to_timestamp("endTime", "yyyy-MM-dd'T'HH:mm:ssX")
                                })
                                .filter(
                                    "value <= 0"
                                )
                                .select("reportUuid", "value", "startTime", "endTime")
                                .show()
)

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 25, Finished, Available, Finished, False)

+--------------------+------------+---------+-------+
|          reportUuid|       value|startTime|endTime|
+--------------------+------------+---------+-------+
|b6a3f4eb-bd5d-404...|-0.027440777|     NULL|   NULL|
|3529abf8-7b5b-4f8...|         0.0|     NULL|   NULL|
|4ef4d706-0332-4fb...|         0.0|     NULL|   NULL|
|2607d601-2c10-41b...|         0.0|     NULL|   NULL|
|595e9076-2826-473...|         0.0|     NULL|   NULL|
|802f3125-ab9f-4bb...|         0.0|     NULL|   NULL|
|7ec63bf4-cdb8-49d...|         0.0|     NULL|   NULL|
|18189cec-6c3a-418...|         0.0|     NULL|   NULL|
|0c6fca51-10f0-456...|         0.0|     NULL|   NULL|
|a6b94319-7032-41c...|         0.0|     NULL|   NULL|
|2a957a0c-3fe6-4c4...|         0.0|     NULL|   NULL|
|ef3b9646-a9ea-430...|         0.0|     NULL|   NULL|
|216d177b-a7ad-43b...|         0.0|     NULL|   NULL|
|475194d5-b316-457...|         0.0|     NULL|   NULL|
|676a1f06-9ec7-45f...|         0.0|     NULL|   NULL|
|6c1eb10e-6e57-4d5...|      

In [22]:
prepared_measures = (measurements_without_enclobject
                                # Conversion des types de colonnes
                                .withColumns(
                                {
                                    "value" : measurements_without_enclobject.value.try_cast("float"),
                                    "startTime" : F.to_timestamp("startTime", "yyyy-MM-dd'T'HH:mm:ss.SSSX"),
                                    "endTime" : F.to_timestamp("endTime", "yyyy-MM-dd'T'HH:mm:ss.SSSX"),
                                    "endLatitude" : F.col("endLatitude").cast("float"),
                                    "endLongitude" : F.col("endLongitude").cast('float'),
                                    "endAltitude" : F.col("endAltitude").cast('float'),
                                    "endAccuracy" : F.col("endAccuracy").cast('float')
                                }
                                )
                                # Filtrer pour enlever les valeurs manquantes
                                .filter("""reportUuid IS NOT NULL 
                                AND dateAndTimeOfCreation IS NOT NULL
                                AND value IS NOT NULL
                                AND startTime IS NOT NULL
                                AND endTime IS NOT NULL
                                AND latitude IS NOT NULL
                                AND longitude IS NOT NULL
                                """)
                                # Filtre pour enlever les valeurs aberrantes
                                .filter(
                                    """
                                    value > 0
                                    AND startTime <= endTime
                                    """
                                )
                                # Mettre valeur de début de position à la valeur de fin de position si nulle
                                .withColumns(
                                    {
                                        "endAltitude" : F.ifnull("endAltitude", "altitude"),
                                        "endLatitude" : F.ifnull("endLatitude", "latitude"),
                                        "endLongitude" : F.ifnull("endLongitude", "longitude"),
                                        "endAccuracy" : F.ifnull("endAccuracy", "accuracy")
                                    }
                                )
                                # Sélectionner les colonnes pour une version fixe
                                .select(
                                    "reportUuid",
                                    "apparatusId",
                                    "apparatusVersion",
                                    "apparatusSensorType",
                                    "apparatusTubeType",
                                    "value",
                                    "hitsNumber",
                                    "calibrationFunction",
                                    "startTime",
                                    "endTime",
                                    "latitude",
                                    "longitude",
                                    "accuracy",
                                    "altitude",
                                    "endLatitude",
                                    "endLongitude",
                                    "endAltitude",
                                    "endAccuracy",
                                    "manualReporting",
                                    "organisationReporting",
                                    "description",
                                    "userId",
                                    "measurementEnvironment",
                                    "departureTime",
                                    "arrivalTime",
                                    "airportOrigin",
                                    "airportDestination",
                                    "aircraftType",
                                    "firstLatitude",
                                    "firstLongitude",
                                    "midLatitude",
                                    "midLongitude",
                                    "lastLatitude",
                                    "lastLongitude",
                                    "dateAndTimeOfCreation",
                                    "reliability",
                                    "qualification",
                                    "atypical"
                                )
                            )

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 26, Finished, Available, Finished, False)

In [23]:
print("Colonnes avant/après : ", 
    len(measurements_without_enclobject.columns), " / ", 
    len(prepared_measures.columns))

print("Lignes avant/après : ", measurements_without_enclobject.count(),"/", prepared_measures.count())

prepared_measures.printSchema()

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 27, Finished, Available, Finished, False)

Colonnes avant/après :  59  /  38
Lignes avant/après :  2031545 / 1854634
root
 |-- reportUuid: string (nullable = true)
 |-- apparatusId: string (nullable = true)
 |-- apparatusVersion: string (nullable = true)
 |-- apparatusSensorType: string (nullable = true)
 |-- apparatusTubeType: string (nullable = true)
 |-- value: float (nullable = true)
 |-- hitsNumber: string (nullable = true)
 |-- calibrationFunction: string (nullable = true)
 |-- startTime: timestamp (nullable = true)
 |-- endTime: timestamp (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- accuracy: double (nullable = true)
 |-- altitude: double (nullable = true)
 |-- endLatitude: double (nullable = true)
 |-- endLongitude: double (nullable = true)
 |-- endAltitude: double (nullable = true)
 |-- endAccuracy: double (nullable = true)
 |-- manualReporting: double (nullable = true)
 |-- organisationReporting: string (nullable = true)
 |-- description: string (nullable = tr

Ensuite, nous allons séparer en deux DataFrames les mesures au sol et au vol.

On peut les filtrer via la colonne qualification.

In [24]:
prepared_measures.show(1)

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 28, Finished, Available, Finished, False)

+--------------------+-----------+----------------+-------------------+-----------------+-----------+----------+--------------------+-------------------+-------------------+----------+---------+--------+--------+-----------+------------+-----------+-----------+---------------+---------------------+-----------+------+----------------------+-------------+-----------+-------------+------------------+------------+-------------+--------------+-----------+------------+------------+-------------+---------------------+-----------+-------------+--------+
|          reportUuid|apparatusId|apparatusVersion|apparatusSensorType|apparatusTubeType|      value|hitsNumber| calibrationFunction|          startTime|            endTime|  latitude|longitude|accuracy|altitude|endLatitude|endLongitude|endAltitude|endAccuracy|manualReporting|organisationReporting|description|userId|measurementEnvironment|departureTime|arrivalTime|airportOrigin|airportDestination|aircraftType|firstLatitude|firstLongitude|midLat

In [25]:
prepared_measures.groupBy("qualification").agg({"reportUuid" : "count"}).show()

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 29, Finished, Available, Finished, False)

+----------------+-----------------+
|   qualification|count(reportUuid)|
+----------------+-----------------+
|           plane|           306942|
|     groundlevel|          1547451|
| temporarysource|              172|
|wrongmeasurement|               69|
+----------------+-----------------+



Les deux types de mesures pour temporarysource et wrongmeasurement ne seront pas à prendre en compte.

In [26]:
# DataFrame sur les mesures au sol
ground_measures = (prepared_measures
                        .filter("qualification = 'groundlevel'")
                        .drop("departureTime", "arrivalTime", "airportOrigin", 
                        "airportDestination", "aircraftType", "firstLatitude",
                        "firstLongitude", "midLatitude", "midLongitude",
                        "lastLatitude", "lastLongitude"))

# DataFrame sur les mesures en vol
flying_measures = (prepared_measures
                        .filter("qualification = 'plane'")
                        .drop("aircraftType"))


StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 30, Finished, Available, Finished, False)

In [27]:
analyser_dataframe(ground_measures)

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 31, Finished, Available, Finished, False)

Nombre de lignes / colonnes 1547451 / 27

--- Structure des colonnes ---
root
 |-- reportUuid: string (nullable = true)
 |-- apparatusId: string (nullable = true)
 |-- apparatusVersion: string (nullable = true)
 |-- apparatusSensorType: string (nullable = true)
 |-- apparatusTubeType: string (nullable = true)
 |-- value: float (nullable = true)
 |-- hitsNumber: string (nullable = true)
 |-- calibrationFunction: string (nullable = true)
 |-- startTime: timestamp (nullable = true)
 |-- endTime: timestamp (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- accuracy: double (nullable = true)
 |-- altitude: double (nullable = true)
 |-- endLatitude: double (nullable = true)
 |-- endLongitude: double (nullable = true)
 |-- endAltitude: double (nullable = true)
 |-- endAccuracy: double (nullable = true)
 |-- manualReporting: double (nullable = true)
 |-- organisationReporting: string (nullable = true)
 |-- description: string (nullable = tru

In [28]:
analyser_dataframe(flying_measures)

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 32, Finished, Available, Finished, False)

Nombre de lignes / colonnes 306942 / 37

--- Structure des colonnes ---
root
 |-- reportUuid: string (nullable = true)
 |-- apparatusId: string (nullable = true)
 |-- apparatusVersion: string (nullable = true)
 |-- apparatusSensorType: string (nullable = true)
 |-- apparatusTubeType: string (nullable = true)
 |-- value: float (nullable = true)
 |-- hitsNumber: string (nullable = true)
 |-- calibrationFunction: string (nullable = true)
 |-- startTime: timestamp (nullable = true)
 |-- endTime: timestamp (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- accuracy: double (nullable = true)
 |-- altitude: double (nullable = true)
 |-- endLatitude: double (nullable = true)
 |-- endLongitude: double (nullable = true)
 |-- endAltitude: double (nullable = true)
 |-- endAccuracy: double (nullable = true)
 |-- manualReporting: double (nullable = true)
 |-- organisationReporting: string (nullable = true)
 |-- description: string (nullable = true

Nous avons donc pu nettoyer les données en enlevant les données aberrantes de nos mesures et séparées les deux types de mesures afin de mener un traitement différent selon leur cas.

On va s'intéresser pour la suite du flux notamment au mesures au sol car les mesures au vol ne vont pas nous intéresser pour l'analyse des radiations en France notamment avec les données météorologiques.

## Ajout de nouvelles colonnes calculées

On va se servir de sources de données mises à disposition par la gouvernement pour identifier à partir de la longitude et latitude le département et la région concernée par ses mesures en France. Ceux en dehors de France seront filtrées et enlevé de notre DataFrame.

Ajouter la durée de mesure entre startTime et endTime.



In [29]:
ground_measures_add_dury = (ground_measures
        .withColumns({"dury" : F.timestamp_diff("SECOND", "startTime", "endTime")})
        )

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 33, Finished, Available, Finished, False)

In [30]:
ground_measures_add_dury.select("latitude", "longitude", "dury").filter("latitude == 51.31956 AND longitude == 6.586638").count()

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 34, Finished, Available, Finished, False)

16871

In [31]:
ground_measures_add_dury.select("latitude", "longitude").show()

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 35, Finished, Available, Finished, False)

+----------+---------+
|  latitude|longitude|
+----------+---------+
|48.2807783|7.6053125|
|48.2807783|7.6053125|
| 50.672861| 3.204542|
|   48.7952|   2.2851|
|   43.1293|    5.955|
|48.2807783|7.6053125|
|48.2807783|7.6053125|
| 50.672861| 3.204542|
|48.2807783|7.6053125|
|   48.7952|   2.2851|
|48.2807783|7.6053125|
|   48.7596|  2.30016|
| 50.672861| 3.204542|
|48.2807783|7.6053125|
|48.2807783|7.6053125|
| 50.672861| 3.204542|
|   48.7952|   2.2851|
|   43.1293|    5.955|
|48.2807783|7.6053125|
|48.2807783|7.6053125|
+----------+---------+
only showing top 20 rows


Pour identifier les communes en France grâce aux coordonnées, nous pouvons utiliser l'API public disponible sur le [découpage administratif du gouvernement](https://geo.api.gouv.fr/decoupage-administratif/communes).

La requête à utiliser serait :

'https://geo.api.gouv.fr/communes?lat={latitude}&lon={longitude}&fields=code,nom,region,departement'

Il ne manque plus qu'à l'appliquer sur chaque ligne et ensuite de filtrer les mesures qui ne sont pas en France.

Changement de façon de faire :

Nous allons changer la façon de joindre les communes, etc de la France en utilisant plutôt plusieurs sources de données GeoJSON disponible dans leur [site](https://www.data.gouv.fr/datasets/contours-administratifs)

In [32]:
# Capturer les coordonnées distinctes
df_coords = (
    ground_measures_add_dury
    .select("latitude", "longitude")
    .dropDuplicates(["latitude", "longitude"])
)

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 36, Finished, Available, Finished, False)

In [33]:
df_france = df_coords.filter(
    (
        F.col("latitude").between(41.3, 51.2) &
        F.col("longitude").between(-5.2, 9.6)
    )
    |
    # Guadeloupe
    (
        F.col("latitude").between(15.8, 16.6) &
        F.col("longitude").between(-61.9, -60.9)
    )
    |
    # Martinique
    (
        F.col("latitude").between(14.3, 14.9) &
        F.col("longitude").between(-61.3, -60.7)
    )
    |
    # Guyane
    (
        F.col("latitude").between(2.0, 6.0) &
        F.col("longitude").between(-54.7, -51.5)
    )
    |
    # La Réunion
    (
        F.col("latitude").between(-21.5, -20.8) &
        F.col("longitude").between(55.2, 55.9)
    )
    |
    # Mayotte
    (
        F.col("latitude").between(-13.1, -12.5) &
        F.col("longitude").between(45.0, 45.4)
    )
)
df_france.cache()

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 37, Finished, Available, Finished, False)

DataFrame[latitude: double, longitude: double]

In [34]:
df_france.count()

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 38, Finished, Available, Finished, False)

410218

In [35]:
print("Nombre de coordonnées distinctes dont il faut faire l'appel API : ", df_coords.count())

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 39, Finished, Available, Finished, False)

Nombre de coordonnées distinctes dont il faut faire l'appel API :  762719


Cela permettra d'éviter de faire plusieurs fois l'appel sur les mêmes coordonnées.

In [36]:
# Juste pour tester la requête
df_coords_fr_limit = df_france.limit(1000)

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 40, Finished, Available, Finished, False)

In [37]:
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType
)

# Schéma du résultat API
schema_api = StructType([
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("code_commune", StringType(), True),
    StructField("nom_commune", StringType(), True),
    StructField("code_departement", StringType(), True),
    StructField("nom_departement", StringType(), True),
    StructField("code_region", StringType(), True),
    StructField("nom_region", StringType(), True),
    StructField("statut_api", StringType(), True)
])

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 41, Finished, Available, Finished, False)

In [38]:
def enrich_partition(rows):

    session = requests.Session()

    # Retry en cas d'erreur temporaire
    retry = Retry(
        total=3,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"]
    )

    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)

    for row in rows:

        lat = float(row["latitude"])
        lon = float(row["longitude"])

        url = "https://geo.api.gouv.fr/communes"

        params = {
            "lat": lat,
            "lon": lon,
            "fields": "code,nom,departement,region"
        }

        try:
            response = session.get(
                url,
                params=params,
                timeout=(5, 15)
            )

            response.raise_for_status()

            data = response.json()

            if data and len(data) > 0:

                commune = data[0]

                departement = commune.get("departement") or {}
                region = commune.get("region") or {}

                yield (
                    lat,
                    lon,
                    commune.get("code"),
                    commune.get("nom"),
                    departement.get("code"),
                    departement.get("nom"),
                    region.get("code"),
                    region.get("nom"),
                    "FOUND"
                )

            else:
                yield (
                    lat, lon,
                    None, None,
                    None, None,
                    None, None,
                    "NOT_FOUND"
                )

        except requests.RequestException:
            yield (
                lat, lon,
                None, None,
                None, None,
                None, None,
                "API_ERROR"
            )

        except (ValueError, TypeError, KeyError):
            yield (
                lat, lon,
                None, None,
                None, None,
                None, None,
                "API_ERROR"
            )

        # Limitation indicative pour gérer le temps d'attente de requête entre deux
        time.sleep(0.0001)

    session.close()

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 42, Finished, Available, Finished, False)

In [39]:
# Réduire le nombre de partitions pour maîtriser
# le nombre de requêtes simultanées.
df_coords_part = df_coords_fr_limit.repartition(100)

# Appels HTTP distribués sur les partitions Spark
rdd_result = df_coords_part.rdd.mapPartitions(enrich_partition)

df_api = spark.createDataFrame(
    rdd_result,
    schema=schema_api
)
df_api.cache()

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 43, Finished, Available, Finished, False)

DataFrame[latitude: double, longitude: double, code_commune: string, nom_commune: string, code_departement: string, nom_departement: string, code_region: string, nom_region: string, statut_api: string]

In [40]:
print("Observation du nombre de mesure selon le statut de l'API : ")
df_api.groupBy("statut_api").agg(F.count("statut_api").alias("effectif")).show()

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 44, Finished, Available, Finished, False)

Observation du nombre de mesure selon le statut de l'API : 
+----------+--------+
|statut_api|effectif|
+----------+--------+
|     FOUND|     918|
| NOT_FOUND|      82|
+----------+--------+



Nous l'appliquerons sur une tâche de spark à part pour toutes les lignes de données.

Essayons avec les fichiers GeoJSON :

https://contours-administratifs.s3.rbx.io.cloud.ovh.net/2026/geojson/communes-1000m.geojson

https://contours-administratifs.s3.rbx.io.cloud.ovh.net/2026/geojson/departements-1000m.geojson

https://contours-administratifs.s3.rbx.io.cloud.ovh.net/2026/geojson/regions-1000m.geojson


In [ ]:
df_france

StatementMeta(, a664742c-887c-4804-8cfa-7753cc28ffda, -1, Cancelled, , Cancelled, True)

In [43]:
import sys

from sedona.spark import SedonaContext

# Informations sur l'environnement
print("Spark :", spark.version)
print("Scala :", spark._jvm.scala.util.Properties.versionNumberString())
print("Java :", spark._jvm.java.lang.System.getProperty("java.version"))
print("Python :", sys.version)

# Initialisation Sedona
sedona = SedonaContext.create(spark)

# Test géospatial
sedona.sql("""
    SELECT ST_AsText(
        ST_Point(2.3522, 48.8566)
    ) AS geometrie
""").show(truncate=False)

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 48, Finished, Available, Finished, False)

Spark : 4.1.1.5.5.20260929.238183312
Scala : 2.13.17
Java : 21.0.13.0.2
Python : 3.13.11 | packaged by conda-forge | (main, Jan 26 2026, 23:57:06) [GCC 14.3.0]
+----------------------+
|geometrie             |
+----------------------+
|POINT (2.3522 48.8566)|
+----------------------+



In [50]:
GEOJSON_PATH = "Files/FranceGeo/"

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 55, Finished, Available, Finished, False)

In [52]:
df_communes = (
    sedona.read
    .format("geojson")
    .load(GEOJSON_PATH+"/communes-1000m.geojson")
)

df_departements = (
    sedona.read
    .format("geojson")
    .load(GEOJSON_PATH+"/departements-1000m.geojson")
)

df_regions = (
    sedona.read
    .format("geojson")
    .load(GEOJSON_PATH+"/regions-1000m.geojson")
)

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 57, Finished, Available, Finished, False)

In [53]:
df_communes.show()

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 59, Finished, Available, Finished, False)

+--------------------+-----------------+
|            features|             type|
+--------------------+-----------------+
|[{POLYGON ((4.958...|FeatureCollection|
+--------------------+-----------------+



In [54]:
df_communes.selectExpr("explode(features) as feature").printSchema()

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 61, Finished, Available, Finished, False)

root
 |-- feature: struct (nullable = true)
 |    |-- geometry: geometry (nullable = true)
 |    |-- properties: struct (nullable = true)
 |    |    |-- code: string (nullable = true)
 |    |    |-- commune: string (nullable = true)
 |    |    |-- departement: string (nullable = true)
 |    |    |-- epci: string (nullable = true)
 |    |    |-- nom: string (nullable = true)
 |    |    |-- plm: boolean (nullable = true)
 |    |    |-- region: string (nullable = true)
 |    |-- type: string (nullable = true)



In [77]:
# Extrayons de notre liste nos lignes de données
df_communes_exploded = (
    df_communes
    .select(F.explode("features").alias("feature"))
)

# Sélectionnons les colonnes qu'on veut de nos features
df_communes_geo = df_communes_exploded.select(
    F.col("feature.geometry").alias("geom_commune"),
    F.col("feature.properties.code").alias("code_commune"),
    F.col("feature.properties.nom").alias("nom_commune"),
    F.col("feature.properties.region").alias("code_region"),
    F.col("feature.properties.departement").alias("code_departement"),
).withColumn("geom_commune", F.expr("ST_MakeValid(geom_commune)"))

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 109, Finished, Available, Finished, False)

In [78]:
df_departements_geo = (
    df_departements
    .select(F.explode("features").alias("feature"))
    .select(
        F.col("feature.geometry").alias("geom_departement"),
        F.col("feature.properties.code").alias("code_departement"),
        F.col("feature.properties.nom").alias("nom_departement")
    ).withColumn("geom_departement", F.expr("ST_MakeValid(geom_departement)"))
)

df_regions_geo = (
    df_regions
    .select(F.explode("features").alias("feature"))
    .select(
        F.col("feature.geometry").alias("geom_region"),
        F.col("feature.properties.code").alias("code_region"),
        F.col("feature.properties.nom").alias("nom_region")
    ).withColumn("geom_region", F.expr("ST_MakeValid(geom_region)"))
)

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 110, Finished, Available, Finished, False)

In [79]:
df_departements_geo.show()

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 111, Finished, Available, Finished, False)

+--------------------+----------------+--------------------+
|    geom_departement|code_departement|     nom_departement|
+--------------------+----------------+--------------------+
|POLYGON ((5.825 4...|              01|                 Ain|
|POLYGON ((3.987 4...|              02|               Aisne|
|POLYGON ((2.678 4...|              03|              Allier|
|POLYGON ((6.728 4...|              04|Alpes-de-Haute-Pr...|
|POLYGON ((6.883 4...|              05|        Hautes-Alpes|
|POLYGON ((6.832 4...|              06|     Alpes-Maritimes|
|POLYGON ((4.809 4...|              07|             Ardèche|
|POLYGON ((5.011 4...|              08|            Ardennes|
|MULTIPOLYGON (((0...|              09|              Ariège|
|POLYGON ((3.745 4...|              10|                Aube|
|POLYGON ((2.895 4...|              11|                Aude|
|POLYGON ((1.916 4...|              12|             Aveyron|
|MULTIPOLYGON (((5...|              13|    Bouches-du-Rhône|
|POLYGON ((0.286 4...|  

In [80]:
df_regions_geo.show()

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 113, Finished, Available, Finished, False)

+--------------------+-----------+--------------------+
|         geom_region|code_region|          nom_region|
+--------------------+-----------+--------------------+
|MULTIPOLYGON (((-...|         01|          Guadeloupe|
|POLYGON ((-61.053...|         02|          Martinique|
|POLYGON ((-52.168...|         03|              Guyane|
|POLYGON ((55.322 ...|         04|          La Réunion|
|MULTIPOLYGON (((4...|         06|             Mayotte|
|POLYGON ((2.52 48...|         11|       Île-de-France|
|POLYGON ((2.571 4...|         24| Centre-Val de Loire|
|POLYGON ((4.622 4...|         27|Bourgogne-Franche...|
|POLYGON ((0.286 4...|         28|           Normandie|
|POLYGON ((3.987 4...|         32|     Hauts-de-France|
|POLYGON ((4.752 5...|         44|           Grand Est|
|MULTIPOLYGON (((-...|         52|    Pays de la Loire|
|MULTIPOLYGON (((-...|         53|            Bretagne|
|MULTIPOLYGON (((-...|         75|  Nouvelle-Aquitaine|
|MULTIPOLYGON (((-...|         76|           Occ

In [81]:
df_communes_geo.show()

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 115, Finished, Available, Finished, False)

+--------------------+------------+--------------------+-----------+----------------+
|        geom_commune|code_commune|         nom_commune|code_region|code_departement|
+--------------------+------------+--------------------+-----------+----------------+
|POLYGON ((4.958 4...|       01001|L'Abergement-Clém...|         84|              01|
|POLYGON ((5.43 45...|       01002|L'Abergement-de-V...|         84|              01|
|POLYGON ((5.409 4...|       01004|   Ambérieu-en-Bugey|         84|              01|
|POLYGON ((4.943 4...|       01005| Ambérieux-en-Dombes|         84|              01|
|POLYGON ((5.571 4...|       01006|             Ambléon|         84|              01|
|POLYGON ((5.424 4...|       01007|            Ambronay|         84|              01|
|POLYGON ((5.342 4...|       01008|            Ambutrix|         84|              01|
|POLYGON ((5.648 4...|       01009|    Andert-et-Condon|         84|              01|
|POLYGON ((5.816 4...|       01010|           Anglefor

In [63]:
df_france.show()

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 78, Finished, Available, Finished, False)

+-------------------+-------------------+
|           latitude|          longitude|
+-------------------+-------------------+
|  48.80213166666667|  2.286426666666667|
| 46.551334398641885|0.41129733188279455|
|         46.3626344|          0.6923943|
| 48.800893333333335| 2.2866133333333334|
|  48.77176333333333|            2.27366|
|          50.353195| 3.5132250000000003|
|          50.353435| 3.5143666666666666|
|  46.86607166666667|          -0.892415|
|        48.06265165|        -3.15865306|
|           43.75512|            7.47042|
| 47.337808333333335|           2.927315|
|  47.43989666666667|-0.5114416666666667|
| 43.626460141737525| 3.8147371480301837|
|         -21.055225|           55.54674|
|-21.060186666666667|          55.542655|
|-21.136116666666666|           55.63169|
|         46.2308912|          6.0546843|
|-21.282038333333333|  55.79582333333333|
|-21.044066666666666|          55.241145|
|  43.53278047597075|  5.376684072746813|
+-------------------+-------------

In [82]:
df_points = (
    df_france
    .withColumn(
        "geom_point",
        F.expr("""
            ST_Point(
                CAST(longitude AS DOUBLE),
                CAST(latitude AS DOUBLE)
            )
        """)
    )
)

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 117, Finished, Available, Finished, False)

In [83]:
df_points.show()

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 118, Finished, Available, Finished, False)

+-------------------+-------------------+--------------------+
|           latitude|          longitude|          geom_point|
+-------------------+-------------------+--------------------+
|  48.80213166666667|  2.286426666666667|POINT (2.28642666...|
| 46.551334398641885|0.41129733188279455|POINT (0.41129733...|
|         46.3626344|          0.6923943|POINT (0.6923943 ...|
| 48.800893333333335| 2.2866133333333334|POINT (2.28661333...|
|  48.77176333333333|            2.27366|POINT (2.27366 48...|
|          50.353195| 3.5132250000000003|POINT (3.51322500...|
|          50.353435| 3.5143666666666666|POINT (3.51436666...|
|  46.86607166666667|          -0.892415|POINT (-0.892415 ...|
|        48.06265165|        -3.15865306|POINT (-3.1586530...|
|           43.75512|            7.47042|POINT (7.47042 43...|
| 47.337808333333335|           2.927315|POINT (2.927315 4...|
|  47.43989666666667|-0.5114416666666667|POINT (-0.5114416...|
| 43.626460141737525| 3.8147371480301837|POINT (3.81473

In [84]:
df_france_communes = (
    df_points.alias("f")
    .join(
        df_communes_geo.alias("c"),
        F.expr("ST_Intersects(f.geom_point, c.geom_commune)"),
        "left"
    )
    .select(
        "f.*",
        F.col("c.code_commune"),
        F.col("c.nom_commune")
    )
)

df_france_departements = (
    df_france_communes.alias("f")
    .join(
        df_departements_geo.alias("d"),
        F.expr("ST_Intersects(f.geom_point, d.geom_departement)"),
        "left"
    )
    .select(
        "f.*",
        F.col("d.code_departement"),
        F.col("d.nom_departement")
    )
)

df_france_enriched = (
    df_france_departements.alias("f")
    .join(
        df_regions_geo.alias("r"),
        F.expr("ST_Intersects(f.geom_point, r.geom_region)"),
        "left"
    )
    .select(
        "f.*",
        F.col("r.code_region"),
        F.col("r.nom_region")
    )
    .filter("r.nom_region IS NOT NULL")
)

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 119, Finished, Available, Finished, False)

In [85]:
df_france_enriched.show()

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 121, Finished, Available, Finished, False)

+-------------------+-------------------+--------------------+------------+--------------------+----------------+----------------+-----------+--------------------+
|           latitude|          longitude|          geom_point|code_commune|         nom_commune|code_departement| nom_departement|code_region|          nom_region|
+-------------------+-------------------+--------------------+------------+--------------------+----------------+----------------+-----------+--------------------+
|  48.80213166666667|  2.286426666666667|POINT (2.28642666...|       92020|           Châtillon|              92|  Hauts-de-Seine|         11|       Île-de-France|
| 46.551334398641885|0.41129733188279455|POINT (0.41129733...|       86157|  Mignaloux-Beauvoir|              86|          Vienne|         75|  Nouvelle-Aquitaine|
|         46.3626344|          0.6923943|POINT (0.6923943 ...|       86190|              Persac|              86|          Vienne|         75|  Nouvelle-Aquitaine|
| 48.80089333333

In [89]:
df_france_enriched = df_france_enriched.cache()

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 128, Finished, Available, Finished, False)

In [90]:
df_france_enriched.show()

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 130, Finished, Available, Finished, False)

+-------------------+-------------------+--------------------+------------+--------------------+----------------+----------------+-----------+--------------------+
|           latitude|          longitude|          geom_point|code_commune|         nom_commune|code_departement| nom_departement|code_region|          nom_region|
+-------------------+-------------------+--------------------+------------+--------------------+----------------+----------------+-----------+--------------------+
|  48.80213166666667|  2.286426666666667|POINT (2.28642666...|       92020|           Châtillon|              92|  Hauts-de-Seine|         11|       Île-de-France|
| 46.551334398641885|0.41129733188279455|POINT (0.41129733...|       86157|  Mignaloux-Beauvoir|              86|          Vienne|         75|  Nouvelle-Aquitaine|
|         46.3626344|          0.6923943|POINT (0.6923943 ...|       86190|              Persac|              86|          Vienne|         75|  Nouvelle-Aquitaine|
| 48.80089333333

In [91]:
# Dernière jointure avec les mesures au sol
ground_measures_french = (
    ground_measures_add_dury
    .alias("f")
    .join(
        df_france_enriched.alias("fr"),
        ["latitude", "longitude"],
        "inner"
    )
    .select(
        "f.*",
        F.col("fr.code_commune"),
        F.col("fr.nom_commune"),
        F.col("fr.code_departement"),
        F.col("fr.nom_departement"),
        F.col("fr.code_region"),
        F.col("fr.nom_region")
    )
)


StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 132, Finished, Available, Finished, False)

In [93]:
ground_measures_french.show()

StatementMeta(, ff43b194-e23c-486b-8512-857f99f37edf, 136, Finished, Available, Finished, False)

+-------------------+------------------+--------------------+----------------+----------------+-------------------+-----------------+-----+----------+-------------------+-------------------+-------------------+--------+--------+-------------------+------------------+-----------+-----------+---------------+---------------------+-----------+------+----------------------+---------------------+-----------+-------------+--------+----+------------+--------------------+----------------+---------------+-----------+----------+
|           latitude|         longitude|          reportUuid|     apparatusId|apparatusVersion|apparatusSensorType|apparatusTubeType|value|hitsNumber|calibrationFunction|          startTime|            endTime|accuracy|altitude|        endLatitude|      endLongitude|endAltitude|endAccuracy|manualReporting|organisationReporting|description|userId|measurementEnvironment|dateAndTimeOfCreation|reliability|qualification|atypical|dury|code_commune|         nom_commune|code_depa

In [ ]:
print("Nombre de lignes : ", ground_measures_french.count())

# Mise en place de la pipeline

Ici, ça sera un résumé de la pipeline à mettre en place pour créer les tables nettoyées des mesures au sol et en vol.

Elle sera composée en médaillon (raw, silver, gold) avec plusieurs transformations à mettre en place explicitée dans ce notebook.

Après avoir exploré un peu les données, voici les transformations à appliquer sur chacun de ces stages :
- raw : ingestion de la source de données extraite du fichier compressé
- silver : nettoyage et transformations des données, séparation des deux types de mesures 
- gold : création des tables analytiques pour le rapport Power BI sur ces mesures (avec DBT) et une table des mesures au sol qui sont en France

Pourquoi ne pas l'exécuter dans ce notebook ?

L'objectif de ce notebook était tout simplement d'explorer et faire des tests de transformations afin de voir la meilleure façon d'explorer ces données.

La mise en place d'une architecture médaillon nous permettra aussi d'offrir d'autre axes d'analyses sans devoir répéter les mêmes transformations.
